# Image Captioning with MS-COCO — fixed submission notebook

This version is cleaned up for the CSE 25 final project expectations. Main fixes:

- train/validation split is by **image id**, not by caption row, avoiding leakage.
- vocabulary is built from **training captions only**.
- BLEU compares each generated caption against **all available human captions** for that image.
- includes multiple controlled experiments: baseline LSTM, lower vocabulary threshold, GRU decoder, and lower learning rate.
- saves checkpoints, loss curves, generated examples, and a CSV results table.

Default uses `val2017` to keep compute reasonable. Change `COCO_SPLIT = "train"` if you download and want to train on full COCO train2017.

## 1. Imports and setup

In [1]:
# Install once if needed
!pip -q install pycocotools nltk

import os, json, random, time
from dataclasses import dataclass, asdict
from collections import Counter, defaultdict

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as T
from torchvision import models

import nltk
nltk.download("punkt", quiet=True)
nltk.download("punkt_tab", quiet=True)
from nltk.tokenize import word_tokenize
from nltk.translate.bleu_score import corpus_bleu, SmoothingFunction

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available(): torch.cuda.manual_seed_all(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("PyTorch:", torch.__version__)
print("Using device:", device)

PyTorch: 2.11.0+cu128
Using device: cuda


## 2. Project settings

In [ ]:
# Portable paths: works on Colab AND a local Jupyter notebook.
# On Colab we write to the fast local VM disk (/content); locally we write under
# the current working directory so there's no PermissionError on '/content'.
try:
    import google.colab  # noqa: F401
    BASE_DIR = "/content"
except ImportError:
    BASE_DIR = os.path.abspath(".")

DATA_DIR = os.path.join(BASE_DIR, "data", "coco")
OUTPUT_DIR = os.path.join(BASE_DIR, "outputs")
os.makedirs(DATA_DIR, exist_ok=True)
os.makedirs(OUTPUT_DIR, exist_ok=True)

COCO_SPLIT = "val"     # "val" for smaller project run; "train" for full dataset if downloaded
IMAGE_SIZE = 224
MAX_CAPTION_LEN = 50
NUM_WORKERS = 2
FAST_DEV_RUN = False   # set True only while debugging; False for final reported results
EVAL_NUM_BATCHES = 5 if FAST_DEV_RUN else 10  # use None for full validation evaluation

print("Data dir:", DATA_DIR)
print("Output dir:", OUTPUT_DIR)
print("Split:", COCO_SPLIT + "2017")
print("Fast dev run:", FAST_DEV_RUN)

## 3. Download MS-COCO data (to local disk)

The dataset is downloaded and extracted with pure Python (`urllib` + `zipfile`), so this runs unchanged on Colab or a local Jupyter notebook. Files land under `DATA_DIR`. The existence checks skip the work if the data is already there. On a persistent local machine the data stays put between sessions; on Colab the VM disk is wiped when the runtime recycles, so a fresh session re-downloads — for `val2017` (~1 GB) that's only a couple of minutes.

In [ ]:
import urllib.request, zipfile

ANNOTATIONS_URL = "http://images.cocodataset.org/annotations/annotations_trainval2017.zip"
if COCO_SPLIT == "val":
    IMAGES_URL = "http://images.cocodataset.org/zips/val2017.zip"
else:
    IMAGES_URL = "http://images.cocodataset.org/zips/train2017.zip"

ann_dir = os.path.join(DATA_DIR, "annotations")
img_dir = os.path.join(DATA_DIR, f"{COCO_SPLIT}2017")

def _download_and_extract(url, zip_path, extract_to):
    """Pure-Python download + unzip (no shell magics, so it works in any Jupyter)."""
    def _progress(block_num, block_size, total_size):
        if total_size > 0:
            pct = min(100, block_num * block_size * 100 / total_size)
            print(f"\r  downloading... {pct:5.1f}%", end="")
    print("Downloading", url)
    urllib.request.urlretrieve(url, zip_path, _progress)
    print("\n  extracting...")
    with zipfile.ZipFile(zip_path, "r") as zf:
        zf.extractall(extract_to)
    os.remove(zip_path)
    print("  done.")

# Download + extract straight to disk. The existence checks skip the work if the
# data is already present (e.g. a re-run within the same session).
if not os.path.exists(ann_dir):
    _download_and_extract(ANNOTATIONS_URL, os.path.join(DATA_DIR, "annotations.zip"), DATA_DIR)
else:
    print("Annotations already extracted.")

if not os.path.exists(img_dir):
    _download_and_extract(IMAGES_URL, os.path.join(DATA_DIR, f"{COCO_SPLIT}2017.zip"), DATA_DIR)
else:
    print(f"{COCO_SPLIT}2017 images already extracted.")

## 4. Load captions and inspect dataset

In [ ]:
annotations_file = os.path.join(DATA_DIR, "annotations", f"captions_{COCO_SPLIT}2017.json")
img_dir = os.path.join(DATA_DIR, f"{COCO_SPLIT}2017")
with open(annotations_file, "r") as f:
    coco_data = json.load(f)

img_id_to_filename = {img["id"]: img["file_name"] for img in coco_data["images"]}
img_id_to_captions = defaultdict(list)
for ann in coco_data["annotations"]:
    img_id_to_captions[ann["image_id"]].append(ann["caption"])

print("Images:", len(coco_data["images"]))
print("Captions:", len(coco_data["annotations"]))
print("Avg captions/image:", round(len(coco_data["annotations"]) / len(coco_data["images"]), 2))

sample_ids = random.sample(list(img_id_to_filename.keys()), 3)
fig, axes = plt.subplots(1, 3, figsize=(16, 5))
for ax, img_id in zip(axes, sample_ids):
    img = Image.open(os.path.join(img_dir, img_id_to_filename[img_id])).convert("RGB")
    ax.imshow(img)
    ax.set_title("\n".join(img_id_to_captions[img_id][:2]), fontsize=8, wrap=True)
    ax.axis("off")
plt.tight_layout(); plt.show()

## 5. Correct image-level train/validation split

In [ ]:
def make_image_id_split(image_ids, train_fraction=0.90, seed=42):
    """Split by image ids so the same image cannot appear in both train and validation."""
    image_ids = list(image_ids)
    rng = random.Random(seed)
    rng.shuffle(image_ids)
    split_idx = int(train_fraction * len(image_ids))
    return set(image_ids[:split_idx]), set(image_ids[split_idx:])

train_img_ids, val_img_ids = make_image_id_split(img_id_to_filename.keys(), 0.90, SEED)
train_annotations = [a for a in coco_data["annotations"] if a["image_id"] in train_img_ids]
val_annotations = [a for a in coco_data["annotations"] if a["image_id"] in val_img_ids]

assert train_img_ids.isdisjoint(val_img_ids)
print("Train images:", len(train_img_ids), " Val images:", len(val_img_ids))
print("Train captions:", len(train_annotations), " Val captions:", len(val_annotations))

## 6. Vocabulary built from training captions only

In [ ]:
class Vocabulary:
    """Maps words to ids and ids back to words."""
    def __init__(self, freq_threshold=5):
        self.freq_threshold = freq_threshold
        self.word2idx = {"<pad>": 0, "<start>": 1, "<end>": 2, "<unk>": 3}
        self.idx2word = {0: "<pad>", 1: "<start>", 2: "<end>", 3: "<unk>"}
        self.word_count = Counter()

    def build_vocabulary(self, captions):
        for cap in captions:
            self.word_count.update(word_tokenize(cap.lower()))
        idx = 4
        for word, count in self.word_count.items():
            if count >= self.freq_threshold:
                self.word2idx[word] = idx
                self.idx2word[idx] = word
                idx += 1

    def numericalize(self, caption):
        ids = [self.word2idx["<start>"]]
        ids += [self.word2idx.get(tok, self.word2idx["<unk>"]) for tok in word_tokenize(caption.lower())]
        ids.append(self.word2idx["<end>"])
        return ids

    def decode(self, ids):
        words = []
        for idx in ids:
            word = self.idx2word.get(int(idx), "<unk>")
            if word == "<end>": break
            if word not in ("<start>", "<pad>"):
                words.append(word)
        return " ".join(words)

    def tokenize_reference(self, caption):
        return word_tokenize(caption.lower())

    def __len__(self): return len(self.word2idx)

def build_vocab_from_train(train_annotations, freq_threshold):
    vocab = Vocabulary(freq_threshold)
    vocab.build_vocabulary([a["caption"] for a in train_annotations])
    return vocab

vocab = build_vocab_from_train(train_annotations, freq_threshold=5)
print("Vocab size:", len(vocab))
print("Sample decoded:", vocab.decode(vocab.numericalize(train_annotations[0]["caption"])))

## 7. Dataset and DataLoader

In [ ]:
class CocoImageCaptionDataset(Dataset):
    """COCO image-caption dataset returning image, caption tensor, and image_id."""
    def __init__(self, img_dir, annotations, img_id_to_filename, vocab, transform=None, max_len=50):
        self.img_dir = img_dir
        self.annotations = list(annotations)
        self.img_id_to_filename = dict(img_id_to_filename)
        self.vocab = vocab
        self.transform = transform
        self.max_len = max_len

    def __len__(self): return len(self.annotations)

    def __getitem__(self, idx):
        ann = self.annotations[idx]
        img_id = ann["image_id"]
        path = os.path.join(self.img_dir, self.img_id_to_filename[img_id])
        image = Image.open(path).convert("RGB")
        if self.transform: image = self.transform(image)
        cap_ids = self.vocab.numericalize(ann["caption"])
        if len(cap_ids) > self.max_len:
            cap_ids = cap_ids[:self.max_len - 1] + [self.vocab.word2idx["<end>"]]
        return image, torch.tensor(cap_ids, dtype=torch.long), img_id

def caption_collate_fn(batch):
    images, captions, image_ids = zip(*batch)
    images = torch.stack(images, dim=0)
    lengths = torch.tensor([len(c) for c in captions], dtype=torch.long)
    max_len = int(lengths.max())
    padded = torch.zeros(len(captions), max_len, dtype=torch.long)
    for i, cap in enumerate(captions):
        padded[i, :len(cap)] = cap
    return images, padded, lengths, torch.tensor(image_ids, dtype=torch.long)

transform = T.Compose([
    T.Resize((256, 256)),
    T.CenterCrop(IMAGE_SIZE),
    T.ToTensor(),
    T.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

def make_loaders(vocab, batch_size=32, num_workers=2):
    train_ds = CocoImageCaptionDataset(img_dir, train_annotations, img_id_to_filename, vocab, transform, MAX_CAPTION_LEN)
    val_ds = CocoImageCaptionDataset(img_dir, val_annotations, img_id_to_filename, vocab, transform, MAX_CAPTION_LEN)
    train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True, num_workers=num_workers,
                              collate_fn=caption_collate_fn, pin_memory=torch.cuda.is_available())
    val_loader = DataLoader(val_ds, batch_size=batch_size, shuffle=False, num_workers=num_workers,
                            collate_fn=caption_collate_fn, pin_memory=torch.cuda.is_available())
    return train_loader, val_loader

train_loader, val_loader = make_loaders(vocab, batch_size=32, num_workers=NUM_WORKERS)
images, captions, lengths, image_ids = next(iter(train_loader))
print("Image batch:", images.shape)
print("Caption batch:", captions.shape)
print("Decoded example:", vocab.decode(captions[0].tolist()))

## 8. Encoder and decoder architectures

In [ ]:
class EncoderCNN(nn.Module):
    """Frozen ResNet-50 image encoder with trainable projection layer."""
    def __init__(self, embed_size):
        super().__init__()
        resnet = models.resnet50(weights=models.ResNet50_Weights.DEFAULT)
        self.resnet = nn.Sequential(*list(resnet.children())[:-1])
        self.linear = nn.Linear(resnet.fc.in_features, embed_size)
        self.bn = nn.BatchNorm1d(embed_size)
        for p in self.resnet.parameters():
            p.requires_grad = False

    def forward(self, images):
        with torch.no_grad():
            features = self.resnet(images)
        features = features.view(features.size(0), -1)
        return self.bn(self.linear(features))

class DecoderRNN(nn.Module):
    """Caption decoder. rnn_type can be 'lstm' or 'gru'."""
    def __init__(self, embed_size, hidden_size, vocab_size, num_layers=1, rnn_type="lstm", dropout=0.3):
        super().__init__()
        self.rnn_type = rnn_type.lower()
        self.embed = nn.Embedding(vocab_size, embed_size)
        self.dropout = nn.Dropout(dropout)
        if self.rnn_type == "lstm":
            self.rnn = nn.LSTM(embed_size, hidden_size, num_layers, batch_first=True)
        elif self.rnn_type == "gru":
            self.rnn = nn.GRU(embed_size, hidden_size, num_layers, batch_first=True)
        else:
            raise ValueError("rnn_type must be 'lstm' or 'gru'")
        self.linear = nn.Linear(hidden_size, vocab_size)

    def forward(self, features, captions):
        embeddings = self.dropout(self.embed(captions[:, :-1]))
        image_step = features.unsqueeze(1)
        inputs = torch.cat((image_step, embeddings), dim=1)
        hiddens, _ = self.rnn(inputs)
        return self.linear(hiddens)

    def generate(self, features, max_len=20, vocab=None):
        self.eval()
        ids, states = [], None
        inputs = features.unsqueeze(1)
        with torch.no_grad():
            for _ in range(max_len):
                hiddens, states = self.rnn(inputs, states)
                logits = self.linear(hiddens.squeeze(1))
                pred = logits.argmax(dim=1)
                token_id = int(pred.item())
                ids.append(token_id)
                if vocab is not None and vocab.idx2word.get(token_id) == "<end>":
                    break
                inputs = self.embed(pred).unsqueeze(1)
        return ids

def count_trainable_parameters(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)

_tmp_enc = EncoderCNN(256).to(device)
_tmp_dec = DecoderRNN(256, 512, len(vocab), rnn_type="lstm").to(device)
print("Trainable encoder params:", count_trainable_parameters(_tmp_enc))
print("Trainable decoder params:", count_trainable_parameters(_tmp_dec))
del _tmp_enc, _tmp_dec
if torch.cuda.is_available(): torch.cuda.empty_cache()

## 9. Training, inference, BLEU, and checkpoint functions

In [ ]:
def train_one_epoch(encoder, decoder, loader, optimizer, criterion, device, epoch_num=0, log_every=100, max_batches=None):
    """Demo Function 2: one epoch of teacher-forced training."""
    encoder.train(); decoder.train()
    running_loss, total_steps = 0.0, 0
    start = time.time()
    for batch_idx, (images, captions, lengths, image_ids) in enumerate(loader):
        if max_batches is not None and batch_idx >= max_batches: break
        images, captions = images.to(device), captions.to(device)
        features = encoder(images)
        outputs = decoder(features, captions)
        # forward() prepends the image and drops the caption's last token internally,
        # so the FULL caption is the target: out[t] predicts captions[t]. (Using
        # captions[:, 1:] here would mismatch the output length by one.)
        targets = captions
        loss = criterion(outputs.reshape(-1, outputs.size(-1)), targets.reshape(-1))
        optimizer.zero_grad(); loss.backward()
        torch.nn.utils.clip_grad_norm_(list(decoder.parameters()) + list(encoder.linear.parameters()) + list(encoder.bn.parameters()), 5.0)
        optimizer.step()
        running_loss += float(loss.item()); total_steps += 1
        if (batch_idx + 1) % log_every == 0:
            print(f"Epoch {epoch_num+1}, Batch {batch_idx+1}/{len(loader)}, Loss {running_loss/total_steps:.4f}, Time {time.time()-start:.1f}s")
    return running_loss / max(total_steps, 1)

def generate_caption(image_path, encoder, decoder, vocab, transform, device, max_len=20, show_image=True):
    """Demo Function 3: generate a caption for one image."""
    encoder.eval(); decoder.eval()
    img = Image.open(image_path).convert("RGB")
    tensor = transform(img).unsqueeze(0).to(device)
    with torch.no_grad():
        features = encoder(tensor)
        ids = decoder.generate(features, max_len=max_len, vocab=vocab)
    caption = vocab.decode(ids)
    if show_image:
        plt.figure(figsize=(6, 5)); plt.imshow(img); plt.title(caption, wrap=True); plt.axis("off"); plt.show()
    return caption

def evaluate_bleu(encoder, decoder, loader, vocab, device, img_id_to_captions, max_len=20, num_batches=None):
    """BLEU-4 against all human references for each image."""
    encoder.eval(); decoder.eval()
    references, hypotheses, seen = [], [], set()
    smoothing = SmoothingFunction().method1
    with torch.no_grad():
        for batch_idx, (images, captions, lengths, image_ids) in enumerate(loader):
            if num_batches is not None and batch_idx >= num_batches: break
            images = images.to(device)
            features = encoder(images)
            for j in range(images.size(0)):
                img_id = int(image_ids[j].item())
                if img_id in seen: continue
                seen.add(img_id)
                hyp_ids = decoder.generate(features[j].unsqueeze(0), max_len=max_len, vocab=vocab)
                hypotheses.append(vocab.decode(hyp_ids).split())
                references.append([vocab.tokenize_reference(c) for c in img_id_to_captions[img_id]])
    return 0.0 if not hypotheses else corpus_bleu(references, hypotheses, smoothing_function=smoothing)

def save_checkpoint(path, encoder, decoder, vocab, config, losses, bleu4):
    torch.save({
        "encoder_state_dict": encoder.state_dict(),
        "decoder_state_dict": decoder.state_dict(),
        "vocab_word2idx": vocab.word2idx,
        "vocab_idx2word": vocab.idx2word,
        "vocab_freq_threshold": vocab.freq_threshold,
        "config": asdict(config),
        "train_losses": losses,
        "bleu4": bleu4,
        "seed": SEED,
        "split": COCO_SPLIT,
    }, path)
    print("Saved checkpoint:", path)

## 10. Experiments — hyperparameter tuning by randomized grid search

We define a grid over the hyperparameters that matter for this model — vocabulary frequency threshold, decoder cell type (LSTM vs GRU), learning rate, hidden size, embedding size, dropout, and number of RNN layers. The full grid is the product of all value lists (192 configurations here), which is far too many to train.

So we do a **randomized grid search**: randomly sample `MAX_TRIALS` configurations from the grid (seeded for reproducibility) and score each by validation BLEU-4. This explores the wide space while keeping compute bounded. To stay within roughly one full-training budget on a T4/L4 GPU, each sampled trial trains on a **subset** (`GRID_EPOCHS` epochs, `GRID_MAX_TRAIN_BATCHES` batches/epoch). The single best-scoring configuration is then **retrained on the full training set** in the next section.

Tuning knobs: widen `PARAM_GRID` to search more values, raise `MAX_TRIALS` for more trials (set it to `None` to run the entire grid), or raise `GRID_EPOCHS` / `GRID_MAX_TRAIN_BATCHES` for a stronger signal per trial.

In [ ]:
import itertools, random as _random

@dataclass
class ExperimentConfig:
    name: str
    freq_threshold: int = 5
    embed_size: int = 256
    hidden_size: int = 512
    num_layers: int = 1
    rnn_type: str = "lstm"
    dropout: float = 0.3
    learning_rate: float = 3e-4
    batch_size: int = 32
    epochs: int = 5
    max_train_batches: int = None   # cap batches/epoch (None = use the full train set)

# ---- Hyperparameter grid ---------------------------------------------------
# All the knobs we search over. The full search space is the product of these
# lists (here 2*2*3*2*2*2*2 = 192 configurations).
PARAM_GRID = {
    "freq_threshold": [3, 5],          # vocabulary cutoff (3 -> larger vocab)
    "rnn_type":       ["lstm", "gru"], # decoder cell type
    "learning_rate":  [1e-3, 3e-4, 1e-4],
    "hidden_size":    [256, 512],      # RNN hidden state size
    "embed_size":     [256, 512],      # word + image embedding size
    "dropout":        [0.3, 0.5],
    "num_layers":     [1, 2],          # stacked RNN layers
}

# Training all 192 combinations is infeasible, so we do a RANDOMIZED grid search:
# randomly sample MAX_TRIALS configs from the grid (seeded for reproducibility).
# This explores the wide space while keeping compute bounded. Set None for full grid.
MAX_TRIALS = 12

# Each sampled trial trains on a SUBSET (fewer epochs + capped batches/epoch) so the
# whole search stays within roughly one full-training budget; the winning config is
# retrained on the full data afterwards. Raise these for a stronger search.
GRID_EPOCHS = 1 if FAST_DEV_RUN else 2
GRID_MAX_TRAIN_BATCHES = 3 if FAST_DEV_RUN else 150   # batches/epoch per trial
GRID_BATCH_SIZE = 32

_grid_keys = list(PARAM_GRID.keys())
_all_combos = list(itertools.product(*PARAM_GRID.values()))
_full_n = len(_all_combos)

_sampler = _random.Random(SEED)
_sampler.shuffle(_all_combos)
_chosen = _all_combos if MAX_TRIALS is None else _all_combos[:MAX_TRIALS]

EXPERIMENTS = []
for _i, _combo in enumerate(_chosen):
    _p = dict(zip(_grid_keys, _combo))
    _name = (f"t{_i:02d}_ft{_p['freq_threshold']}_{_p['rnn_type']}"
             f"_lr{_p['learning_rate']:g}_h{_p['hidden_size']}_e{_p['embed_size']}"
             f"_d{_p['dropout']}_L{_p['num_layers']}")
    EXPERIMENTS.append(ExperimentConfig(
        name=_name,
        batch_size=GRID_BATCH_SIZE,
        epochs=GRID_EPOCHS,
        max_train_batches=GRID_MAX_TRAIN_BATCHES,
        **_p,
    ))

print(f"Full grid: {_full_n} configurations")
print(f"Training {len(EXPERIMENTS)} sampled trial(s) "
      f"({GRID_EPOCHS} epochs x up to {GRID_MAX_TRAIN_BATCHES} batches each)")
pd.DataFrame([asdict(cfg) for cfg in EXPERIMENTS])

## 11. Run the grid search (and retrain the best config)

The first cell trains every grid combination on the subset and writes `experiment_results.csv`, sorted by BLEU-4. The second cell retrains the top configuration on the full training set and selects it as the model used for the qualitative examples.

In [ ]:
import copy

# ---- Early-stopping settings -----------------------------------------------
# We monitor validation BLEU-4 after each epoch. If it doesn't improve by at
# least EARLY_STOP_MIN_DELTA for EARLY_STOP_PATIENCE consecutive epochs, we stop
# and roll back to the best epoch's weights.
EARLY_STOP_PATIENCE = 3
EARLY_STOP_MIN_DELTA = 1e-3


def train_one_epoch(encoder, decoder, train_loader, optimizer, criterion, device, epoch, max_batches=None):
    encoder.train()
    decoder.train()

    total_loss = 0.0
    num_batches = 0

    for batch_idx, batch in enumerate(train_loader):
        if max_batches is not None and batch_idx >= max_batches:
            break

        # Handles loaders that return (images, captions) or extra values
        images = batch[0].to(device)
        captions = batch[1].to(device)

        # DecoderRNN.forward() prepends the image feature and internally drops the
        # caption's last token (it embeds captions[:, :-1]). So we pass the FULL
        # caption as both the input and the target: out[t] is trained to predict
        # captions[t] (out[0], produced from the image, predicts <start>).
        # Passing captions[:, :-1] here would slice a second time -> off-by-one.
        targets = captions

        optimizer.zero_grad()

        features = encoder(images)
        outputs = decoder(features, captions)

        # Defensive alignment in case lengths ever differ.
        min_len = min(outputs.size(1), targets.size(1))
        outputs = outputs[:, :min_len, :]
        targets = targets[:, :min_len]

        loss = criterion(
            outputs.reshape(-1, outputs.size(-1)),
            targets.reshape(-1)
        )

        loss.backward()
        optimizer.step()

        total_loss += loss.item()
        num_batches += 1

    return total_loss / max(num_batches, 1)


def run_experiment(config):
    print("\n" + "="*80)
    print("Running:", config.name)
    print("="*80)

    exp_vocab = build_vocab_from_train(train_annotations, config.freq_threshold)
    exp_train_loader, exp_val_loader = make_loaders(exp_vocab, config.batch_size, NUM_WORKERS)

    encoder = EncoderCNN(config.embed_size).to(device)
    decoder = DecoderRNN(
        config.embed_size,
        config.hidden_size,
        len(exp_vocab),
        config.num_layers,
        config.rnn_type,
        config.dropout
    ).to(device)

    criterion = nn.CrossEntropyLoss(ignore_index=exp_vocab.word2idx["<pad>"])

    optimizer = torch.optim.Adam(
        list(decoder.parameters()) +
        list(encoder.linear.parameters()) +
        list(encoder.bn.parameters()),
        lr=config.learning_rate
    )

    scheduler = torch.optim.lr_scheduler.StepLR(
        optimizer,
        step_size=3,
        gamma=0.8
    )

    # Subset cap: FAST_DEV_RUN forces a tiny run; otherwise use the config's
    # per-trial cap (set for grid search, None for full retraining).
    max_batches = 3 if FAST_DEV_RUN else config.max_train_batches

    ckpt_path = os.path.join(OUTPUT_DIR, f"{config.name}.pt")

    losses, val_bleus = [], []
    best_bleu_epoch = -1.0
    best_epoch = -1
    best_enc_state, best_dec_state = None, None
    epochs_no_improve = 0
    start = time.time()

    for epoch in range(config.epochs):
        loss = train_one_epoch(
            encoder,
            decoder,
            exp_train_loader,
            optimizer,
            criterion,
            device,
            epoch,
            max_batches=max_batches
        )
        losses.append(loss)
        scheduler.step()

        # Validate this epoch (capped by EVAL_NUM_BATCHES to keep it cheap).
        val_bleu = evaluate_bleu(
            encoder, decoder, exp_val_loader, exp_vocab, device,
            img_id_to_captions, num_batches=EVAL_NUM_BATCHES
        )
        val_bleus.append(val_bleu)

        improved = val_bleu > best_bleu_epoch + EARLY_STOP_MIN_DELTA
        flag = ""
        if improved:
            best_bleu_epoch = val_bleu
            best_epoch = epoch
            epochs_no_improve = 0
            # Snapshot the best weights so far and persist them to disk.
            best_enc_state = copy.deepcopy(encoder.state_dict())
            best_dec_state = copy.deepcopy(decoder.state_dict())
            save_checkpoint(ckpt_path, encoder, decoder, exp_vocab, config, losses, val_bleu)
            flag = "  <-- best so far (checkpoint saved)"
        else:
            epochs_no_improve += 1

        print(f"Epoch {epoch+1}/{config.epochs}: train loss = {loss:.4f}, "
              f"val BLEU-4 = {val_bleu:.4f}{flag}")

        if epochs_no_improve >= EARLY_STOP_PATIENCE:
            print(f"Early stopping: no val BLEU-4 improvement for "
                  f"{EARLY_STOP_PATIENCE} epoch(s). Best was epoch {best_epoch+1} "
                  f"(BLEU-4 = {best_bleu_epoch:.4f}).")
            break

    # Roll back to the best epoch's weights for the returned/used model.
    if best_enc_state is not None:
        encoder.load_state_dict(best_enc_state)
        decoder.load_state_dict(best_dec_state)

    bleu4 = best_bleu_epoch if best_epoch >= 0 else (val_bleus[-1] if val_bleus else 0.0)

    plt.figure(figsize=(6, 4))
    plt.plot(range(1, len(losses) + 1), losses, marker="o", label="train loss")
    plt.plot(range(1, len(val_bleus) + 1), val_bleus, marker="s", label="val BLEU-4")
    plt.title("Training: " + config.name)
    plt.xlabel("Epoch")
    plt.ylabel("loss / BLEU-4")
    plt.legend()
    plt.grid(True, alpha=0.3)
    plt.tight_layout()

    loss_path = os.path.join(OUTPUT_DIR, f"{config.name}_loss.png")
    plt.savefig(loss_path, dpi=150)
    plt.show()

    result = {
        **asdict(config),
        "vocab_size": len(exp_vocab),
        "final_train_loss": losses[-1],
        "best_epoch": best_epoch + 1,
        "epochs_trained": len(losses),
        "bleu4": bleu4,
        "train_time_seconds": round(time.time() - start, 2),
        "checkpoint": ckpt_path,
        "loss_curve": loss_path,
        "encoder_trainable_params": count_trainable_parameters(encoder),
        "decoder_trainable_params": count_trainable_parameters(decoder)
    }

    return result, encoder, decoder, exp_vocab, exp_val_loader


results, best_model, best_bleu = [], None, -1

for cfg in EXPERIMENTS:
    result, encoder, decoder, exp_vocab, exp_val_loader = run_experiment(cfg)
    results.append(result)

    if result["bleu4"] > best_bleu:
        best_bleu = result["bleu4"]
        best_model = (encoder, decoder, exp_vocab, exp_val_loader, cfg.name)
    else:
        del encoder, decoder

        if torch.cuda.is_available():
            torch.cuda.empty_cache()

        if torch.backends.mps.is_available():
            torch.mps.empty_cache()

results_df = pd.DataFrame(results).sort_values("bleu4", ascending=False)

results_csv = os.path.join(OUTPUT_DIR, "experiment_results.csv")
results_df.to_csv(results_csv, index=False)

print("Saved results:", results_csv)
results_df

In [ ]:
# ---- Retrain the winning grid configuration on the FULL training set -------
# The grid above only trained each config on a small subset. Here we take the
# best-scoring hyperparameters and train them properly (full data, no batch cap)
# so the qualitative section below uses a well-trained model. We set a high epoch
# ceiling and let early stopping (in run_experiment) end training once the
# validation BLEU-4 plateaus, keeping the best epoch's weights.
best = results_df.iloc[0]
print("Best grid config:", best["name"], "| grid BLEU-4:", round(float(best["bleu4"]), 4))

final_config = ExperimentConfig(
    name="best_" + str(best["name"]),
    freq_threshold=int(best["freq_threshold"]),
    embed_size=int(best["embed_size"]),
    hidden_size=int(best["hidden_size"]),
    num_layers=int(best["num_layers"]),
    rnn_type=str(best["rnn_type"]),
    dropout=float(best["dropout"]),
    learning_rate=float(best["learning_rate"]),
    batch_size=int(best["batch_size"]),
    epochs=15,                # upper bound; early stopping ends it sooner if val BLEU-4 plateaus
    max_train_batches=None,   # use the entire training set
)

final_result, f_enc, f_dec, f_vocab, f_val_loader = run_experiment(final_config)
results.append(final_result)

# The fully trained model should drive the qualitative examples.
best_bleu = final_result["bleu4"]
best_model = (f_enc, f_dec, f_vocab, f_val_loader, final_config.name)
print("Selected model for qualitative section:", final_config.name,
      "| BLEU-4 =", round(best_bleu, 4))

results_df = pd.DataFrame(results).sort_values("bleu4", ascending=False)
results_df.to_csv(results_csv, index=False)
results_df

In [ ]:
# ---- Save the best model's weights to Google Drive (persistent storage) -----
# OUTPUT_DIR lives on the ephemeral VM disk, so checkpoints are LOST when the
# Colab runtime recycles. This cell mounts Drive and copies the best model's
# checkpoint (plus the results CSV) into a folder on your Drive so the weights
# persist across sessions. On a local Jupyter (no Colab) it just reports the
# local path, since local disk already persists.
import shutil

# Folder on your Drive where the best weights will be saved. Change if you like.
DRIVE_SAVE_DIR = "/content/drive/MyDrive/image_captioning"

# The best model's checkpoint was written by run_experiment() to OUTPUT_DIR.
best_name = best_model[4]
best_ckpt_path = os.path.join(OUTPUT_DIR, f"{best_name}.pt")

try:
    from google.colab import drive
    drive.mount("/content/drive")            # prompts for authorization once
    os.makedirs(DRIVE_SAVE_DIR, exist_ok=True)

    dest_ckpt = os.path.join(DRIVE_SAVE_DIR, f"{best_name}.pt")
    shutil.copy2(best_ckpt_path, dest_ckpt)
    print("Saved best weights to Drive:", dest_ckpt)

    # Also copy the results table for convenience.
    if os.path.exists(results_csv):
        shutil.copy2(results_csv, os.path.join(DRIVE_SAVE_DIR, "experiment_results.csv"))
        print("Saved results CSV to Drive:", os.path.join(DRIVE_SAVE_DIR, "experiment_results.csv"))
except ImportError:
    print("Not on Colab — weights already persist locally at:",
          os.path.abspath(best_ckpt_path))

## 12. Qualitative examples from best model

In [ ]:
encoder, decoder, best_vocab, best_val_loader, best_name = best_model
print("Best model:", best_name)

sample_eval_ids = random.sample(list(val_img_ids), min(6, len(val_img_ids)))
fig, axes = plt.subplots(2, 3, figsize=(16, 9)); axes = axes.flatten()
rows = []
for ax, img_id in zip(axes, sample_eval_ids):
    fname = img_id_to_filename[img_id]
    path = os.path.join(img_dir, fname)
    pred = generate_caption(path, encoder, decoder, best_vocab, transform, device, show_image=False)
    refs = img_id_to_captions[img_id]
    img = Image.open(path).convert("RGB")
    ax.imshow(img); ax.set_title(pred, fontsize=9, wrap=True); ax.axis("off")
    rows.append({"image_id": img_id, "file_name": fname, "generated_caption": pred,
                 "reference_caption_1": refs[0], "reference_caption_2": refs[1] if len(refs) > 1 else ""})
plt.suptitle("Generated captions: " + best_name); plt.tight_layout()
qual_path = os.path.join(OUTPUT_DIR, "generated_caption_examples.png")
plt.savefig(qual_path, dpi=150, bbox_inches="tight"); plt.show()

qual_df = pd.DataFrame(rows)
qual_csv = os.path.join(OUTPUT_DIR, "qualitative_examples.csv")
qual_df.to_csv(qual_csv, index=False)
print("Saved examples:", qual_path)
print("Saved qualitative table:", qual_csv)
qual_df

## 13. Required demo function summary

In [ ]:
def load_coco_dataset(data_dir=DATA_DIR, split=COCO_SPLIT, freq_threshold=5, batch_size=32, num_workers=2):
    """Demo Function 1: load dataset, split by image id, build vocabulary, return loaders and stats."""
    ann_file = os.path.join(data_dir, "annotations", f"captions_{split}2017.json")
    im_dir = os.path.join(data_dir, f"{split}2017")
    with open(ann_file, "r") as f: raw = json.load(f)
    id_to_file = {img["id"]: img["file_name"] for img in raw["images"]}
    tr_ids, vl_ids = make_image_id_split(id_to_file.keys(), 0.90, SEED)
    tr_anns = [a for a in raw["annotations"] if a["image_id"] in tr_ids]
    vl_anns = [a for a in raw["annotations"] if a["image_id"] in vl_ids]
    voc = build_vocab_from_train(tr_anns, freq_threshold)
    tr_ds = CocoImageCaptionDataset(im_dir, tr_anns, id_to_file, voc, transform, MAX_CAPTION_LEN)
    vl_ds = CocoImageCaptionDataset(im_dir, vl_anns, id_to_file, voc, transform, MAX_CAPTION_LEN)
    tr_loader = DataLoader(tr_ds, batch_size=batch_size, shuffle=True, num_workers=num_workers, collate_fn=caption_collate_fn)
    vl_loader = DataLoader(vl_ds, batch_size=batch_size, shuffle=False, num_workers=num_workers, collate_fn=caption_collate_fn)
    stats = {"train_images": len(tr_ids), "val_images": len(vl_ids), "train_captions": len(tr_anns),
             "val_captions": len(vl_anns), "vocab_size": len(voc), "freq_threshold": freq_threshold}
    return tr_loader, vl_loader, voc, stats

_demo_train, _demo_val, _demo_vocab, _demo_stats = load_coco_dataset(freq_threshold=5, batch_size=16)
print("Demo Function 1 stats:", _demo_stats)
print("Demo Function 2: train_one_epoch(...) is defined above")
print("Demo Function 3: generate_caption(...) is defined above")

## 14. Report-ready notes

In [ ]:
print("Files created in outputs/:")
for fname in sorted(os.listdir(OUTPUT_DIR)):
    print(" -", fname)
print("\nUse experiment_results.csv for the report's quantitative table.")
print("Use *_loss.png for training curves.")
print("Use generated_caption_examples.png and qualitative_examples.csv for qualitative discussion.")
print("Limitations to mention: smaller val2017 split, frozen encoder, greedy decoding, and limited epochs due to compute.")

In [ ]:
import shutil
# Portable archiving (works in any Jupyter, not just Colab's shell).
archive_path = shutil.make_archive("image_captioning_outputs", "zip", OUTPUT_DIR)
print("Created:", archive_path)

In [ ]:
# Trigger a browser download on Colab; on a local Jupyter the file is already on
# disk next to the notebook, so we just print its location.
try:
    from google.colab import files
    files.download("image_captioning_outputs.zip")
except ImportError:
    print("Saved locally:", os.path.abspath("image_captioning_outputs.zip"))